## Lets infer a model from the Hub

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
"microsoft/Phi-3-mini-4k-instruct",
device_map="cuda",
torch_dtype="auto",
trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

In [ ]:
'''
return_full_text > By setting this to False , the prompt will not be returned but
merely the output of the model.
max_new_tokens > The maximum number of tokens the model will generate. By
setting a limit, we prevent long and unwieldy output as some
models might continue generating output until they reach their context window.
do_sample > Whether the model uses a sampling strategy to choose the
next token. By setting this to False , the model will always
select the next most probable token'''

from transformers import pipeline
# Create a pipeline
generator = pipeline(
"text-generation",
model=model,
tokenizer=tokenizer,
return_full_text=False,
max_new_tokens=500,
do_sample=False
)

In [ ]:
response = generator("what is egypt's capital ?")
print(response[0]['generated_text'])

In [6]:
import transformers
transformers.__version__

c:\Users\Amr osama abdellatif\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'4.44.0'

#### Using a System msg

detailed approach

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

# Define a custom system message
system_message = """You are a helpful, harmless, and precise AI assistant that provides accurate information and never makes things up."""

# Create messages in the chat format
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": "Tell me about quantum computing"}
]

# Format the messages using the model's chat template
prompt = tokenizer.apply_chat_template(messages, tokenize=False)
print(prompt)
# Generate response
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    inputs["input_ids"],
    max_new_tokens=500,
    do_sample=False
)

# Decode and print response
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

pipeline can handle this details without seperating each component

In [ ]:
from transformers import pipeline

# Create pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

# Define system message and user prompt
system_message = """You are a helpful, harmless, and precise AI assistant that provides accurate information and never makes things up."""
user_prompt = "Tell me about quantum computing"

# Format using chat template
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# Generate response
response = generator(prompt)
print(response[0]['generated_text'])

## Naive Rag : dense retrieval

### lets use ollama

In [ ]:
!pip install ollama chromadb PyPDF2

In [1]:
import ollama
import chromadb
import PyPDF2
import os
import re
import uuid

c:\Users\Amr osama abdellatif\AppData\Local\Programs\Python\Python312\Lib\site-packages\onnxruntime\capi\onnxruntime_validation.py:26: UserWarning: Unsupported Windows version (11). ONNX Runtime supports Windows 10 and above, only.
  warnings.warn(


In [2]:
def extract_pdf_text(pdf_path):
    """Extract text from a PDF file - simple version"""
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page in pdf_reader.pages:
                text += page.extract_text() + "\n"
    except:
        print(f"Could not read {pdf_path}")
    return text

pdf_file = "1706.03762v7.pdf"  
if os.path.exists(pdf_file):
    raw_text = extract_pdf_text(pdf_file)
    print(f"Extracted {len(raw_text)} characters from PDF")
else:
    # Use sample text for testing
    raw_text = """
    Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
    Deep learning uses neural networks with multiple layers to model complex patterns in data.
    Natural language processing enables computers to understand and generate human language.
    Computer vision allows machines to interpret and understand visual information from images.
    Artificial intelligence is transforming many industries through automation and intelligent decision making.
    """
    print("Using sample text since no PDF found")

Extracted 39487 characters from PDF


In [3]:
def clean_text(text):
    """Clean up the text"""
    # Remove extra spaces and weird characters
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,!?-]', '', text)
    return text.strip()

def split_into_chunks(text, chunk_size=200):
    """Split text into smaller chunks"""
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    return chunks

# Clean and split our text
clean_text_content = clean_text(raw_text)
text_chunks = split_into_chunks(clean_text_content)

print(f"Created {len(text_chunks)} chunks")
for i, chunk in enumerate(text_chunks[:3]):  # Show first 3 chunks
    print(f"\nChunk {i+1}: {chunk[:100]}...")

Created 30 chunks

Chunk 1: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and...

Chunk 2: fraction of the training costs of the best models from the literature. We show that the Transformer ...

Chunk 3: 7 neural networks in particular, have been firmly established as state of the art approaches in sequ...


In [4]:
def get_embedding(text):
    """Get embedding for one piece of text"""
    try:
        response = ollama.embeddings(model="nomic-embed-text", prompt=text)
        return response['embedding']
    except:
        print(f"Error getting embedding for: {text[:50]}...")
        return []

# Get embeddings for all chunks
print("Getting embeddings from Ollama...")
embeddings = []

for i, chunk in enumerate(text_chunks):
    print(f"Processing chunk {i+1}/{len(text_chunks)}")
    embedding = get_embedding(chunk)
    embeddings.append(embedding)

print(f"Got {len(embeddings)} embeddings")
print(f"Each embedding has {len(embeddings[0])} dimensions")

Getting embeddings from Ollama...
Processing chunk 1/30
Processing chunk 2/30
Processing chunk 3/30
Processing chunk 4/30
Processing chunk 5/30
Processing chunk 6/30
Processing chunk 7/30
Processing chunk 8/30
Processing chunk 9/30
Processing chunk 10/30
Processing chunk 11/30
Processing chunk 12/30
Processing chunk 13/30
Processing chunk 14/30
Processing chunk 15/30
Processing chunk 16/30
Processing chunk 17/30
Processing chunk 18/30
Processing chunk 19/30
Processing chunk 20/30
Processing chunk 21/30
Processing chunk 22/30
Processing chunk 23/30
Processing chunk 24/30
Processing chunk 25/30
Processing chunk 26/30
Processing chunk 27/30
Processing chunk 28/30
Processing chunk 29/30
Processing chunk 30/30
Got 30 embeddings
Each embedding has 768 dimensions


In [ ]:
# Create ChromaDB client and collection
client = chromadb.PersistentClient(path="./simple_chroma_db")

# # Delete collection if it exists (fresh start)
# try:
#     client.delete_collection("simple_rag")
# except:
#     pass


# Create new collection
collection = client.create_collection(
    name="simple_rag")

print("Created ChromaDB collection")



Created ChromaDB collection


In [15]:
# Generate IDs and metadata for each chunk
chunk_ids = [f"chunk_{i}" for i in range(len(text_chunks))]
chunk_metadata = [{"chunk_number": i, "source": "document"} for i in range(len(text_chunks))]

# Add everything to ChromaDB
collection.add(
    documents=text_chunks,
    embeddings=embeddings,
    ids=chunk_ids,
    metadatas=chunk_metadata
)

print(f"Added {len(text_chunks)} chunks to ChromaDB")
print(f"Collection now has {collection.count()} documents")

Added 30 chunks to ChromaDB
Collection now has 30 documents


In [39]:
def search_documents(query, n_results=3):
    """Search for similar documents"""
    # Get embedding for the query
    query_embedding = get_embedding(query)
    
    # Search ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=['documents', 'distances', 'metadatas']
    )
    print(results)    
    return results
    

# Test the search
test_query = "What is attention is all you need ?"
search_results = search_documents(test_query)

print(f"Search results for: '{test_query}'")
print("-" * 50)

for i in range(len(search_results['documents'][0])):
    doc = search_results['documents'][0][i]
    distance = search_results['distances'][0][i]
    similarity = 1 - distance  # Convert distance to similarity
    
    print(f"\nResult {i+1} (Similarity: {similarity:.3f}):")
    print(f"{doc}")

Number of requested results 3 is greater than number of elements in index 2, updating n_results = 2


{'ids': [['chunk_0', 'chunk_1']], 'distances': [[0.5056214568667006, 0.6276617448829913]], 'metadatas': [[{'chunk_number': 0, 'source': 'document'}, {'chunk_number': 1, 'source': 'document'}]], 'embeddings': None, 'documents': [['Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data without being explicitly programmed. There are three main types of machine learning supervised learning uses labeled data to train models, unsupervised learning finds patterns in unlabeled data, and reinforcement learning learns through trial and error with rewards. Deep learning uses neural networks with multiple layers to model complex patterns in data. Each layer learns increasingly abstract features. Natural language processing enables computers to understand and generate human language through techniques like tokenization, parsing, and semantic analysis. Computer vision allows machines to interpret visual information from images using convolutional 

In [ ]:
def generate_answer(query, search_results):
    """Generate answer using retrieved documents"""
    # Get the documents from search results
    documents = search_results['documents'][0]
    
    # Create context from retrieved documents
    context = "\n\n".join([f"Context {i+1}: {doc}" for i, doc in enumerate(documents)])
    
    # Create the prompt
    prompt = f"""Based on the following contexts, answer the question.
    If you can't find the answer in the contexts, say so.

Contexts:
{context}

Question: {query}

Answer:"""
    
    # Get response from Ollama
    response = ollama.chat(
        model="llama3.2:1b",  # or whatever model you have
        messages=[{"role": "system", "content": 'you are a ML expert',
                    "role": "user", "content": prompt}]
    )
    
    return response['message']['content']

# Generate answer for our test query
answer = generate_answer(test_query, search_results)

print(f"Question: {test_query}")
print(f"Answer: {answer}")

Question: What is attention is all you need ?
Answer: Based on the contexts provided, it appears that "attention" is often mentioned in relation to self-attention mechanisms, which are a type of neural network layer used for various tasks such as processing sequential data (e.g., natural language text) and understanding context.

In many of these contexts, attention is described as being necessary or crucial for certain tasks. For example, in Context 1, "but its application should be just - this is what we are missing , in my opinion . EOS pad The Law will never be perfect , but its application should be just - this is what we are missing , in my opinion ." suggests that the law (EOS pad) requires attention to be applied. Similarly, in Context 2, "by layer normalization. We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions." implies that self-attention is necessary for achieving desired results.

In Context 3, "hea

Task : add multiple documents

## Chain Of Thoughts

In [ ]:
!pip install ollama chromadb PyPDF2

In [ ]:
import ollama
import chromadb
import PyPDF2
import os
import re
import uuid

In [24]:
def extract_pdf_text(pdf_path):
    """Extract text from a PDF file - simple version"""
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page in pdf_reader.pages:
                text += page.extract_text() + "\n"
    except:
        print(f"Could not read {pdf_path}")
    return text

# Test it (replace with your PDF path or skip if no PDF)
pdf_file = "your_document.pdf"  # Change this path
if os.path.exists(pdf_file):
    raw_text = extract_pdf_text(pdf_file)
    print(f"Extracted {len(raw_text)} characters from PDF")
else:
    # Use sample text for testing - more detailed for CoT
    raw_text = """
    Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data without being explicitly programmed.
    There are three main types of machine learning: supervised learning uses labeled data to train models, unsupervised learning finds patterns in unlabeled data, and reinforcement learning learns through trial and error with rewards.
    Deep learning uses neural networks with multiple layers to model complex patterns in data. Each layer learns increasingly abstract features.
    Natural language processing enables computers to understand and generate human language through techniques like tokenization, parsing, and semantic analysis.
    Computer vision allows machines to interpret visual information from images using convolutional neural networks and feature detection algorithms.
    Artificial intelligence is transforming industries through automation, predictive analytics, and intelligent decision making systems.
    Neural networks are inspired by biological neurons and consist of interconnected nodes that process information through weighted connections.
    Training a neural network involves adjusting weights through backpropagation to minimize prediction errors on training data.
    """
    print("Using detailed sample text since no PDF found")

Using detailed sample text since no PDF found


In [25]:
def clean_text(text):
    """Clean up the text"""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,!?-]', '', text)
    return text.strip()

def split_into_chunks(text, chunk_size=150):
    """Split text into smaller chunks - smaller for CoT"""
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    return chunks

# Clean and split our text
clean_text_content = clean_text(raw_text)
text_chunks = split_into_chunks(clean_text_content)

print(f"Created {len(text_chunks)} chunks")
for i, chunk in enumerate(text_chunks[:3]):
    print(f"\nChunk {i+1}: {chunk[:100]}...")

Created 2 chunks

Chunk 1: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

Chunk 2: through backpropagation to minimize prediction errors on training data....


In [26]:
def get_embedding(text):
    """Get embedding for one piece of text"""
    try:
        response = ollama.embeddings(model="nomic-embed-text", prompt=text)
        return response['embedding']
    except:
        print(f"Error getting embedding for: {text[:50]}...")
        return []

# Get embeddings for all chunks
print("Getting embeddings from Ollama...")
embeddings = []

for i, chunk in enumerate(text_chunks):
    print(f"Processing chunk {i+1}/{len(text_chunks)}")
    embedding = get_embedding(chunk)
    embeddings.append(embedding)

print(f"Got {len(embeddings)} embeddings")

Getting embeddings from Ollama...
Processing chunk 1/2
Processing chunk 2/2
Got 2 embeddings


In [ ]:
# Create ChromaDB client and collection
client = chromadb.PersistentClient(path="./cot_chroma_db")

# # Delete collection if it exists (fresh start)
# try:
#     client.delete_collection("cot_rag")
# except:
#     pass

# Create new collection
collection = client.create_collection(
    name="cot_rag")

print("Created ChromaDB collection for CoT RAG")

Created ChromaDB collection for CoT RAG


In [28]:
# Generate IDs and metadata
chunk_ids = [f"chunk_{i}" for i in range(len(text_chunks))]
chunk_metadata = [{"chunk_number": i, "source": "document"} for i in range(len(text_chunks))]

# Add everything to ChromaDB
collection.add(
    documents=text_chunks,
    embeddings=embeddings,
    ids=chunk_ids,
    metadatas=chunk_metadata
)

print(f"Added {len(text_chunks)} chunks to ChromaDB")
print(f"Collection now has {collection.count()} documents")

Added 2 chunks to ChromaDB
Collection now has 2 documents


In [31]:
def search_documents(query, n_results=5):
    """Search for relevant documents"""
    query_embedding = get_embedding(query)
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=['documents', 'distances', 'metadatas']
    )
    
    return results


In [34]:
def call_llm(prompt, model="llama3.2:1b"):
    """Make a single LLM call"""
    try:
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}]
        )
        return response['message']['content']
    except Exception as e:
        return f"Error: {e}"

# Test basic functions
test_search = search_documents("What is machine learning?")
print(f"Found {len(test_search['documents'][0])} relevant documents")

Number of requested results 5 is greater than number of elements in index 2, updating n_results = 2


Found 2 relevant documents


In [35]:
def step1_analyze_question(question):
    """Step 1: Analyze what the question is really asking"""
    
    prompt = f"""Analyze this question and break it down:

Question: {question}

Please identify:
1. What is the main topic being asked about?
2. What specific aspects or details are being requested?
3. What type of answer would be most helpful (definition, explanation, comparison, process, etc.)?
4. Are there any sub-questions or related concepts to address?

Provide a clear analysis:"""

    print("🔍 STEP 1: Analyzing the question...")
    analysis = call_llm(prompt)
    print(f"Analysis: {analysis}")
    return analysis

# Test Step 1
test_question = "How does deep learning differ from traditional machine learning?"
question_analysis = step1_analyze_question(test_question)

🔍 STEP 1: Analyzing the question...
Analysis: Analysis of the question:

1. The main topic being asked about is "deep learning" and its differences from traditional machine learning.
2. The specific aspects or details being requested are the fundamental differences between deep learning and traditional machine learning.
3. A most helpful type of answer would be one that provides a clear explanation, definition, comparison, or process-based analysis to facilitate understanding.

Breakdown of the question:

- "Deep learning" refers to a subset of machine learning techniques that use neural networks with multiple layers to learn complex patterns in data. It's an area of artificial intelligence (AI) where computers are trained by mimicking how humans think.
- Traditional machine learning, on the other hand, focuses on simpler algorithms and models designed to make predictions or classify data without explicitly learning a pattern through experience.

Helpful answer types:

1. Definition: A

In [36]:
def step2_evaluate_sources(question, question_analysis, search_results):
    """Step 2: Evaluate which sources are most relevant"""
    
    # Prepare sources text
    sources_text = ""
    for i, doc in enumerate(search_results['documents'][0]):
        sources_text += f"Source {i+1}: {doc}\n\n"
    
    prompt = f"""Based on the question analysis, evaluate these sources:

Original Question: {question}

Question Analysis: {question_analysis}

Available Sources:
{sources_text}

For each source, determine:
1. How relevant is it to answering the question?
2. What specific information does it provide?
3. Which sources should be prioritized?
4. Are there any gaps in the available information?

Provide your source evaluation:"""

    print("📊 STEP 2: Evaluating sources...")
    evaluation = call_llm(prompt)
    print(f"Source Evaluation: {evaluation}")
    return evaluation

# Test Step 2
search_results = search_documents(test_question)
source_evaluation = step2_evaluate_sources(test_question, question_analysis, search_results)

Number of requested results 5 is greater than number of elements in index 2, updating n_results = 2


📊 STEP 2: Evaluating sources...
Source Evaluation: Based on the analysis of the original question and its breakdown, here's an evaluation of each source:

1. **Source 1**:
	* Relevance: 6/10 (While it provides some general information about machine learning, it doesn't specifically address the differences between deep learning and traditional machine learning.)
	* Specificity: 4/10 (It mentions supervised, unsupervised, and reinforcement learning, but doesn't provide an in-depth analysis of these concepts or their applications.)
	* Information provided: The source provides a brief overview of machine learning as a subset of AI.
	* Priority: Should be prioritized for its general information on machine learning.
2. **Source 2**:
	* Relevance: 9/10 (This source explicitly mentions deep learning and traditional machine learning, providing a clear comparison between the two.)
	* Specificity: 8/10 (It provides detailed explanations of neural networks, backpropagation, and their applications 

In [37]:
def step3_synthesize_information(question, question_analysis, source_evaluation, search_results):
    """Step 3: Synthesize information from sources"""
    
    sources_text = ""
    for i, doc in enumerate(search_results['documents'][0]):
        sources_text += f"Source {i+1}: {doc}\n\n"
    
    prompt = f"""Now synthesize the information to build toward an answer:

Question: {question}
Question Analysis: {question_analysis}
Source Evaluation: {source_evaluation}

Sources:
{sources_text}

Based on your analysis and evaluation:
1. What are the key points from the most relevant sources?
2. How do these points relate to each other?
3. What patterns or connections can you identify?
4. What is the logical flow for presenting this information?

Provide your information synthesis:"""

    print("🔗 STEP 3: Synthesizing information...")
    synthesis = call_llm(prompt)
    print(f"Information Synthesis: {synthesis}")
    return synthesis

# Test Step 3
info_synthesis = step3_synthesize_information(test_question, question_analysis, source_evaluation, search_results)

🔗 STEP 3: Synthesizing information...
Information Synthesis: The main difference between deep learning and traditional machine learning lies in their approaches to modeling complex patterns in data.

Traditional machine learning relies on simpler algorithms and models that focus on making predictions or classifications without explicitly learning a pattern through experience. This approach is often limited by the availability of large amounts of training data, which can be difficult to obtain for small datasets. In contrast, deep learning uses neural networks with multiple layers to model complex patterns in data, allowing it to handle high-dimensional data and make accurate predictions.

The key points that emerge from this analysis are:

* Traditional machine learning models rely on simpler algorithms and models, whereas deep learning models use neural networks with many interconnected nodes (neurons) to model complex patterns.
* Deep learning can handle high-dimensional data, wherea

In [38]:
def step4_construct_answer(question, question_analysis, source_evaluation, synthesis):
    """Step 4: Construct the final answer"""
    
    prompt = f"""Now construct the final comprehensive answer:

Original Question: {question}

Your Previous Analysis:
- Question Analysis: {question_analysis}
- Source Evaluation: {source_evaluation}  
- Information Synthesis: {synthesis}

Based on all your previous thinking, provide a complete, well-structured answer that:
1. Directly addresses the original question
2. Uses the most relevant information you identified
3. Follows the logical flow you developed
4. Is clear and comprehensive

Final Answer:"""

    print("✅ STEP 4: Constructing final answer...")
    final_answer = call_llm(prompt)
    print(f"Final Answer: {final_answer}")
    return final_answer

# Test Step 4
final_answer = step4_construct_answer(test_question, question_analysis, source_evaluation, info_synthesis)

✅ STEP 4: Constructing final answer...
Final Answer: ## Step 1: Identify the key differences between deep learning and traditional machine learning.
The key differences between deep learning and traditional machine learning are their approach to handling complex data, the complexity of an approach, and the modeling capabilities.

## Step 2: Determine the relevance of each point in understanding the difference between deep learning and traditional machine learning.
Traditional machine learning relies on simpler algorithms and models that focus on making predictions or classifications without explicitly learning a pattern through experience. In contrast, deep learning uses neural networks with multiple layers to model complex patterns in data.

## Step 3: Explain how these key points relate to each other.
The simplicity or complexity of an approach is reflected in the number of layers and interconnected nodes used in the model. The ability of a machine learning approach to handle complex

## Keyword search

What is BM25? 🤔

BM25 = "Best Matching 25" - it's a keyword scoring algorithm that figures out how relevant a document is to your search query.

The Simple Goal 🎯
BM25's job: Given a search query like "machine learning", score every document from 0 to infinity based on how well it matches. Higher score = better match.

How BM25 Thinks 🧠
BM25 asks 3 simple questions about each document:

1. Term Frequency (TF) - "How often does this word appear?"
2. Document Frequency (DF) - "How rare is this word?"
3. Document Length - "Is this document too long or too short?"

BM25 Score = TF × IDF × Length Penalty




BM25

In [ ]:
!pip install ollama langchain langchain-community PyPDF2 rank-bm25

In [41]:
import ollama
import PyPDF2
import os
import re
from langchain.retrievers import BM25Retriever
from langchain.schema import Document
from rank_bm25 import BM25Okapi

In [42]:
def extract_pdf_text(pdf_path):
    """Extract text from a PDF file"""
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page in pdf_reader.pages:
                text += page.extract_text() + "\n"
    except:
        print(f"Could not read {pdf_path}")
    return text

# Test it
pdf_file = "your_document.pdf"  # Change this path
if os.path.exists(pdf_file):
    raw_text = extract_pdf_text(pdf_file)
    print(f"Extracted {len(raw_text)} characters from PDF")
else:
    # Sample text with good keywords for BM25 testing
    raw_text = """
    Machine learning algorithms learn patterns from data to make predictions without explicit programming.
    Supervised learning uses labeled training data with input-output pairs to train predictive models.
    Unsupervised learning discovers hidden patterns and structures in unlabeled data without target variables.
    Reinforcement learning agents learn optimal actions through trial and error using reward signals.
    Deep learning neural networks have multiple hidden layers for learning complex hierarchical representations.
    Convolutional neural networks excel at image recognition tasks using convolution and pooling operations.
    Recurrent neural networks process sequential data like text and time series using memory cells.
    Natural language processing combines linguistics and machine learning for text understanding and generation.
    Computer vision algorithms analyze digital images and videos to extract meaningful information.
    Classification algorithms predict discrete categories while regression algorithms predict continuous values.
    Clustering algorithms group similar data points together without labeled examples.
    Decision trees create interpretable models using if-then rules for classification and regression.
    Random forests combine multiple decision trees to improve accuracy and reduce overfitting.
    Support vector machines find optimal decision boundaries for classification problems.
    """
    print("Using sample text with good keywords for BM25")

Using sample text with good keywords for BM25


In [43]:
def clean_text(text):
    """Clean up the text"""
    # Keep more words for BM25 - it needs keywords
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,!?-]', '', text)
    return text.strip()

def split_into_chunks(text, chunk_size=100):
    """Split text into chunks - smaller for better keyword matching"""
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    return chunks

# Clean and split our text
clean_text_content = clean_text(raw_text)
text_chunks = split_into_chunks(clean_text_content)

print(f"Created {len(text_chunks)} chunks for BM25")
for i, chunk in enumerate(text_chunks[:3]):
    print(f"\nChunk {i+1}: {chunk}")

Created 2 chunks for BM25

Chunk 1: Machine learning algorithms learn patterns from data to make predictions without explicit programming. Supervised learning uses labeled training data with input-output pairs to train predictive models. Unsupervised learning discovers hidden patterns and structures in unlabeled data without target variables. Reinforcement learning agents learn optimal actions through trial and error using reward signals. Deep learning neural networks have multiple hidden layers for learning complex hierarchical representations. Convolutional neural networks excel at image recognition tasks using convolution and pooling operations. Recurrent neural networks process sequential data like text and time series using memory cells. Natural language processing combines linguistics and machine learning

Chunk 2: for text understanding and generation. Computer vision algorithms analyze digital images and videos to extract meaningful information. Classification algorithms predic

In [44]:
def create_langchain_documents(text_chunks):
    """Convert text chunks to LangChain Document objects"""
    documents = []
    
    for i, chunk in enumerate(text_chunks):
        doc = Document(
            page_content=chunk,
            metadata={
                "chunk_id": i,
                "source": "document",
                "chunk_number": i
            }
        )
        documents.append(doc)
    
    return documents

# Create LangChain documents
langchain_docs = create_langchain_documents(text_chunks)

print(f"Created {len(langchain_docs)} LangChain documents")
print(f"Sample document: {langchain_docs[0].page_content[:100]}...")
print(f"Sample metadata: {langchain_docs[0].metadata}")

Created 2 LangChain documents
Sample document: Machine learning algorithms learn patterns from data to make predictions without explicit programmin...
Sample metadata: {'chunk_id': 0, 'source': 'document', 'chunk_number': 0}


In [45]:
# Create BM25 retriever using LangChain
bm25_retriever = BM25Retriever.from_documents(langchain_docs)

# Set how many documents to retrieve
bm25_retriever.k = 4  # Return top 4 most relevant documents

print("BM25 Retriever created!")
print(f"Configured to return top {bm25_retriever.k} documents")
print(f"Total documents indexed: {len(langchain_docs)}")

BM25 Retriever created!
Configured to return top 4 documents
Total documents indexed: 2


In [46]:
def test_bm25_search(query):
    """Test BM25 search with a query"""
    print(f"🔍 BM25 Search for: '{query}'")
    print("-" * 50)
    
    # Search using BM25
    results = bm25_retriever.get_relevant_documents(query)
    
    print(f"Found {len(results)} relevant documents:")
    
    for i, doc in enumerate(results):
        print(f"\nResult {i+1}:")
        print(f"Content: {doc.page_content}")
        print(f"Metadata: {doc.metadata}")
        print("-" * 30)
    
    return results

# Test BM25 search
test_query = "machine learning algorithms"
search_results = test_bm25_search(test_query)

🔍 BM25 Search for: 'machine learning algorithms'
--------------------------------------------------
Found 2 relevant documents:

Result 1:
Content: Machine learning algorithms learn patterns from data to make predictions without explicit programming. Supervised learning uses labeled training data with input-output pairs to train predictive models. Unsupervised learning discovers hidden patterns and structures in unlabeled data without target variables. Reinforcement learning agents learn optimal actions through trial and error using reward signals. Deep learning neural networks have multiple hidden layers for learning complex hierarchical representations. Convolutional neural networks excel at image recognition tasks using convolution and pooling operations. Recurrent neural networks process sequential data like text and time series using memory cells. Natural language processing combines linguistics and machine learning
Metadata: {'chunk_id': 0, 'source': 'document', 'chunk_number': 0

c:\Users\Amr osama abdellatif\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


In [48]:
def call_ollama(prompt):
    """Simple Ollama call"""
    try:
        response = ollama.chat(
            model="llama3.2:1b",
            messages=[{"role": "user", "content": prompt}]
        )
        return response['message']['content']
    except Exception as e:
        return f"Error: {e}"

def generate_answer_from_bm25(query, bm25_results):
    """Generate answer using BM25 retrieved documents"""
    
    # Combine retrieved documents
    context = ""
    for i, doc in enumerate(bm25_results):
        context += f"Source {i+1}: {doc.page_content}\n\n"
    
    # Create prompt
    prompt = f"""Based on the following sources found through keyword search, answer the question.

Sources:
{context}

Question: {query}

Answer:"""
    
    # Get answer from Ollama
    answer = call_ollama(prompt)
    return answer

# Test answer generation
answer = generate_answer_from_bm25(test_query, search_results)
print(f"\nQuestion: {test_query}")
print(f"BM25 Answer: {answer}")


Question: machine learning algorithms
BM25 Answer: Based on the provided sources, machine learning algorithms are categorized into three main types:

1. **Supervised Learning**: Uses labeled training data with input-output pairs to train predictive models (Source 1).
2. **Unsupervised Learning**: Discovers hidden patterns and structures in unlabeled data without target variables (Source 1).
3. **Reinforcement Learning**: Learns optimal actions through trial and error using reward signals, but does not explicitly program the algorithm (Source 1).

Additionally, other types of machine learning algorithms mentioned include:

* **Deep Learning**: Uses multiple hidden layers for learning complex hierarchical representations (Source 2).
* **Convolutional Neural Networks (CNNs)**: Excel at image recognition tasks using convolution and pooling operations (Source 2).
* **Recurrent Neural Networks (RNNs)**: Process sequential data like text and time series using memory cells (Source 2).
* **Nat

In [49]:
def bm25_rag(question):
    """Complete BM25 RAG pipeline"""
    print(f"📝 BM25 RAG Pipeline")
    print(f"Question: {question}")
    print("=" * 60)
    
    # Step 1: BM25 keyword search
    print("🔍 Step 1: BM25 keyword search...")
    results = bm25_retriever.get_relevant_documents(question)
    
    # Step 2: Show retrieved documents
    print("📄 Step 2: Retrieved documents:")
    for i, doc in enumerate(results):
        print(f"  {i+1}. {doc.page_content[:80]}...")
    
    # Step 3: Generate answer
    print("🤖 Step 3: Generating answer...")
    answer = generate_answer_from_bm25(question, results)
    
    print(f"\n✅ BM25 Answer: {answer}")
    return answer

# Test complete BM25 RAG
bm25_rag("What is supervised learning?")

📝 BM25 RAG Pipeline
Question: What is supervised learning?
🔍 Step 1: BM25 keyword search...
📄 Step 2: Retrieved documents:
  1. for text understanding and generation. Computer vision algorithms analyze digita...
  2. Machine learning algorithms learn patterns from data to make predictions without...
🤖 Step 3: Generating answer...

✅ BM25 Answer: Supervised learning is a type of machine learning algorithm that uses labeled training data with input-output pairs to train predictive models. In other words, it involves teaching a model what data to expect as output based on the given inputs. The model learns from the patterns and relationships in the data, allowing it to make accurate predictions or classifications without needing explicit programming.


'Supervised learning is a type of machine learning algorithm that uses labeled training data with input-output pairs to train predictive models. In other words, it involves teaching a model what data to expect as output based on the given inputs. The model learns from the patterns and relationships in the data, allowing it to make accurate predictions or classifications without needing explicit programming.'

### Task : RRF - Contextual Rag

#### read and implement 

## Agentic Rag - Function calling

In [ ]:
!pip install ollama chromadb PyPDF2

In [1]:
from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
import json
import re

In [3]:
# Initialize Ollama
ollama = Ollama(
    base_url='http://localhost:11434',
    model="llama3.2"
)

# Simple function to call
def calculate_area(length, width):
    """Calculate area of a rectangle"""
    result = length * width
    return f"The area is {result} square units"

def get_weather(city):
    """Mock weather function"""
    weather_data = {
        "New York": "Sunny, 75°F",
        "London": "Rainy, 60°F", 
        "Cairo": "Hot, 95°F"
    }
    return weather_data.get(city, "Weather data not available")

# Function calling template
function_template = """
You can call these functions:
- calculate_area(length, width) - calculates rectangle area
- get_weather(city) - gets weather for a city

Question: {question}

If you need to use a function, respond with:
FUNCTION_CALL: function_name(param1, param2)

Otherwise, just answer normally.
"""

prompt = PromptTemplate(
    input_variables=["question"],
    template=function_template
)

chain = LLMChain(llm=ollama, prompt=prompt)

def process_response(question):
    response = chain.invoke({"question": question})
    answer = response['text']
    
    # Check if LLM wants to call a function
    if "FUNCTION_CALL:" in answer:
        # Extract function call
        match = re.search(r'FUNCTION_CALL:\s*(\w+)\((.*?)\)', answer)
        if match:
            func_name = match.group(1)
            params = match.group(2)
            
            print(f"LLM wants to call: {func_name}({params})")
            
            # Execute the function
            if func_name == "calculate_area":
                # Parse parameters
                nums = [float(x.strip()) for x in params.split(',')]
                result = calculate_area(nums[0], nums[1])
                
            elif func_name == "get_weather":
                city = params.strip().strip('"\'')
                result = get_weather(city)
            
            print(f"Function result: {result}")
            
            # Get final response with function result
            final_template = """
            Question: {question}
            Function result: {function_result}
            
            Please provide a complete answer using this function result.
            """
            
            final_prompt = PromptTemplate(
                input_variables=["question", "function_result"],
                template=final_template
            )
            
            final_chain = LLMChain(llm=ollama, prompt=final_prompt)
            final_response = final_chain.invoke({
                "question": question,
                "function_result": result
            })
            
            return final_response['text']
    
    return answer

# Test the function calling
print("=== Test 1: Area calculation ===")
result1 = process_response("What's the area of a rectangle that is 5 meters long and 3 meters wide?")
print("Final Answer:", result1)

print("\n=== Test 2: Weather query ===")
result2 = process_response("What's the weather like in Cairo?")
print("Final Answer:", result2)

print("\n=== Test 3: Regular question ===")
result3 = process_response("What causes northern lights?")
print("Final Answer:", result3)

=== Test 1: Area calculation ===
LLM wants to call: calculate_area(5, 3)
Function result: The area is 15.0 square units
Final Answer: To calculate the area of a rectangle, you can use the formula:

Area = length x width

In this case, the length of the rectangle is 5 meters and the width is 3 meters.

Area = 5 x 3
= 15.0 square units

So, the area of the rectangle is indeed 15.0 square units.

=== Test 2: Weather query ===
LLM wants to call: get_weather("Cairo")
Function result: Hot, 95°F
Final Answer: I don't have real-time access to current weather conditions. However, I can suggest some options to help you find out the current weather in Cairo.

You can check online weather websites such as AccuWeather, Weather.com, or the National Weather Service (NWS) for the most up-to-date information on weather conditions in Cairo.

Alternatively, you can also check social media or news websites for any updates on the weather in Cairo.

=== Test 3: Regular question ===
LLM wants to call: get_we

In [50]:
import ollama
import chromadb
import PyPDF2
import os
import re
import uuid
import json
import math
from datetime import datetime, timedelta

In [51]:
def extract_pdf_text(pdf_path):
    """Extract text from a PDF file"""
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page in pdf_reader.pages:
                text += page.extract_text() + "\n"
    except:
        print(f"Could not read {pdf_path}")
    return text

# Setup documents
pdf_file = "your_document.pdf"
if os.path.exists(pdf_file):
    raw_text = extract_pdf_text(pdf_file)
    print(f"Extracted {len(raw_text)} characters from PDF")
else:
    # Sample text with some numbers and data for tool usage
    raw_text = """
    Machine learning algorithms require computational resources and data for training.
    A typical neural network might have 1000 parameters and require 100 hours of training time.
    Deep learning models can have millions of parameters, with training costs reaching $50,000 for large models.
    The accuracy of machine learning models typically ranges from 80% to 95% on standard datasets.
    Popular machine learning frameworks include TensorFlow (released in 2015), PyTorch (released in 2016), and Scikit-learn (released in 2007).
    Training a GPT-3 model cost approximately $4.6 million and required 175 billion parameters.
    A standard computer vision model might achieve 92% accuracy on ImageNet with 25 million parameters.
    The data preprocessing phase typically takes 60% of a machine learning project timeline.
    Cloud computing costs for ML training can range from $0.10 to $3.00 per hour depending on GPU type.
    """
    print("Using sample text with numbers for agent tool usage")

# Clean and chunk text
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,!?$%-]', '', text)
    return text.strip()

def split_into_chunks(text, chunk_size=150):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

clean_text_content = clean_text(raw_text)
text_chunks = split_into_chunks(clean_text_content)
print(f"Created {len(text_chunks)} chunks")

Using sample text with numbers for agent tool usage
Created 1 chunks


In [52]:
def get_embedding(text):
    try:
        response = ollama.embeddings(model="nomic-embed-text", prompt=text)
        return response['embedding']
    except:
        print(f"Error getting embedding")
        return []

# Get embeddings
embeddings = []
for i, chunk in enumerate(text_chunks):
    print(f"Processing chunk {i+1}/{len(text_chunks)}")
    embedding = get_embedding(chunk)
    embeddings.append(embedding)

# Setup ChromaDB
client = chromadb.PersistentClient(path="./agent_rag_db")

try:
    client.delete_collection("agent_rag")
except:
    pass

collection = client.create_collection(name="agent_rag", metadata={"hnsw:space": "cosine"})

# Add documents
chunk_ids = [f"chunk_{i}" for i in range(len(text_chunks))]
chunk_metadata = [{"chunk_number": i, "source": "document"} for i in range(len(text_chunks))]

collection.add(
    documents=text_chunks,
    embeddings=embeddings,
    ids=chunk_ids,
    metadatas=chunk_metadata
)

print(f"Vector store ready with {len(text_chunks)} documents")

Processing chunk 1/1
Vector store ready with 1 documents


In [53]:
# Tool 1: Calculator
def calculator(expression):
    """
    Calculate mathematical expressions. 
    Usage: calculator("2 + 2") or calculator("sqrt(16)") or calculator("log(100)")
    """
    try:
        # Replace common math functions
        expression = expression.replace("sqrt", "math.sqrt")
        expression = expression.replace("log", "math.log")
        expression = expression.replace("sin", "math.sin") 
        expression = expression.replace("cos", "math.cos")
        expression = expression.replace("^", "**")  # Power operator
        
        # Safe evaluation
        result = eval(expression, {"__builtins__": {}, "math": math})
        return f"Result: {result}"
    except Exception as e:
        return f"Calculator error: {str(e)}"

# Tool 2: Text Analyzer
def text_analyzer(text):
    """
    Analyze text statistics.
    Usage: text_analyzer("some text to analyze")
    """
    words = text.split()
    chars = len(text)
    sentences = len([s for s in text.split('.') if s.strip()])
    
    return f"""Text Analysis:
- Words: {len(words)}
- Characters: {chars}
- Sentences: {sentences}
- Average word length: {chars/len(words):.1f}
- Longest word: {max(words, key=len) if words else 'None'}"""

# Tool 3: Date Calculator
def date_calculator(operation, days=0):
    """
    Calculate dates. 
    Usage: date_calculator("add", 30) or date_calculator("subtract", 7)
    """
    today = datetime.now()
    
    if operation == "add":
        new_date = today + timedelta(days=days)
        return f"Today + {days} days = {new_date.strftime('%Y-%m-%d')}"
    elif operation == "subtract":
        new_date = today - timedelta(days=days)
        return f"Today - {days} days = {new_date.strftime('%Y-%m-%d')}"
    elif operation == "today":
        return f"Today is {today.strftime('%Y-%m-%d %H:%M')}"
    else:
        return "Usage: date_calculator('add'|'subtract'|'today', days)"

# Tool 4: Number Extractor
def number_extractor(text):
    """
    Extract all numbers from text.
    Usage: number_extractor("I have 5 apples and 3 oranges")
    """
    import re
    numbers = re.findall(r'-?\d+\.?\d*', text)
    if numbers:
        return f"Found numbers: {numbers}"
    else:
        return "No numbers found in text"

# Tool 5: Simple Counter
def counter(items_text):
    """
    Count items in comma-separated text.
    Usage: counter("apple, banana, orange, apple")
    """
    items = [item.strip().lower() for item in items_text.split(',')]
    count_dict = {}
    for item in items:
        count_dict[item] = count_dict.get(item, 0) + 1
    
    result = "Item counts:\n"
    for item, count in sorted(count_dict.items()):
        result += f"- {item}: {count}\n"
    return result

# Tool 6: Document Search (Our RAG retrieval)
def document_search(query):
    """
    Search through documents for relevant information.
    Usage: document_search("machine learning costs")
    """
    query_embedding = get_embedding(query)
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        include=['documents', 'distances', 'metadatas']
    )
    
    if results['documents'] and len(results['documents'][0]) > 0:
        found_docs = []
        for i, doc in enumerate(results['documents'][0]):
            distance = results['distances'][0][i]
            similarity = 1 - distance
            found_docs.append(f"Document {i+1} (relevance: {similarity:.2f}): {doc}")
        
        return "Found relevant documents:\n" + "\n\n".join(found_docs)
    else:
        return "No relevant documents found"

# Create tools registry
AVAILABLE_TOOLS = {
    "calculator": calculator,
    "text_analyzer": text_analyzer,
    "date_calculator": date_calculator, 
    "number_extractor": number_extractor,
    "counter": counter,
    "document_search": document_search
}

print("🛠️ Available Tools:")
for tool_name, tool_func in AVAILABLE_TOOLS.items():
    print(f"- {tool_name}: {tool_func.__doc__.strip().split('.')[0]}")

🛠️ Available Tools:
- calculator: Calculate mathematical expressions
- text_analyzer: Analyze text statistics
- date_calculator: Calculate dates
- number_extractor: Extract all numbers from text
- counter: Count items in comma-separated text
- document_search: Search through documents for relevant information


In [54]:
# Test each tool
print("🧪 Testing Tools:")
print("=" * 40)

print("1. Calculator:")
print(calculator("2 + 2 * 3"))
print(calculator("sqrt(16)"))

print("\n2. Text Analyzer:")
print(text_analyzer("Hello world! This is a test."))

print("\n3. Date Calculator:")
print(date_calculator("today"))
print(date_calculator("add", 30))

print("\n4. Number Extractor:")
print(number_extractor("I spent $25.50 on 3 items"))

print("\n5. Counter:")
print(counter("apple, banana, apple, orange, apple"))

print("\n6. Document Search:")
print(document_search("machine learning training costs"))

Number of requested results 3 is greater than number of elements in index 1, updating n_results = 1


🧪 Testing Tools:
1. Calculator:
Result: 8
Result: 4.0

2. Text Analyzer:
Text Analysis:
- Words: 6
- Characters: 28
- Sentences: 1
- Average word length: 4.7
- Longest word: world!

3. Date Calculator:
Today is 2025-05-22 16:35
Today + 30 days = 2025-06-21

4. Number Extractor:
Found numbers: ['25.50', '3']

5. Counter:
Item counts:
- apple: 3
- banana: 1
- orange: 1


6. Document Search:
Found relevant documents:
Document 1 (relevance: 0.84): Machine learning algorithms require computational resources and data for training. A typical neural network might have 1000 parameters and require 100 hours of training time. Deep learning models can have millions of parameters, with training costs reaching $50,000 for large models. The accuracy of machine learning models typically ranges from 80% to 95% on standard datasets. Popular machine learning frameworks include TensorFlow released in 2015, PyTorch released in 2016, and Scikit-learn released in 2007. Training a GPT-3 model cost approximate

In [57]:
def call_llm(prompt):
    """Call Ollama LLM"""
    try:
        response = ollama.chat(
            model="llama3.2:1b",
            messages=[{"role": "user", "content": prompt}]
        )
        return response['message']['content']
    except Exception as e:
        return f"LLM Error: {e}"

def parse_tool_call(text):
    """Parse tool calls from LLM response"""
    # Look for tool calls in format: TOOL_NAME(arguments)
    import re
    
    # Pattern to match: TOOL_NAME(arguments)
    pattern = r'(\w+)\((.*?)\)'
    matches = re.findall(pattern, text)
    
    tool_calls = []
    for tool_name, args in matches:
        if tool_name.lower() in AVAILABLE_TOOLS:
            tool_calls.append((tool_name.lower(), args))
    
    return tool_calls

def execute_tool(tool_name, args_string):
    """Execute a tool with parsed arguments"""
    try:
        tool_func = AVAILABLE_TOOLS[tool_name]
        
        # Parse arguments based on tool
        if tool_name == "calculator":
            return tool_func(args_string.strip('"\''))
        elif tool_name == "text_analyzer":
            return tool_func(args_string.strip('"\''))
        elif tool_name == "date_calculator":
            # Parse operation and days
            parts = [p.strip().strip('"\'') for p in args_string.split(',')]
            operation = parts[0] if parts else "today"
            days = int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 0
            return tool_func(operation, days)
        elif tool_name == "number_extractor":
            return tool_func(args_string.strip('"\''))
        elif tool_name == "counter":
            return tool_func(args_string.strip('"\''))
        elif tool_name == "document_search":
            return tool_func(args_string.strip('"\''))
        else:
            return f"Unknown tool: {tool_name}"
            
    except Exception as e:
        return f"Tool execution error: {str(e)}"

# Test tool calling
test_response = "I need to calculate 15 * 8 using calculator(15 * 8) and search for info using document_search(neural networks)"
tool_calls = parse_tool_call(test_response)
print(f"Parsed tool calls: {tool_calls}")

for tool_name, args in tool_calls:
    result = execute_tool(tool_name, args)
    print(f"{tool_name}({args}) -> {result}")

Number of requested results 3 is greater than number of elements in index 1, updating n_results = 1


Parsed tool calls: [('calculator', '15 * 8'), ('document_search', 'neural networks')]
calculator(15 * 8) -> Result: 120
document_search(neural networks) -> Found relevant documents:
Document 1 (relevance: 0.63): Machine learning algorithms require computational resources and data for training. A typical neural network might have 1000 parameters and require 100 hours of training time. Deep learning models can have millions of parameters, with training costs reaching $50,000 for large models. The accuracy of machine learning models typically ranges from 80% to 95% on standard datasets. Popular machine learning frameworks include TensorFlow released in 2015, PyTorch released in 2016, and Scikit-learn released in 2007. Training a GPT-3 model cost approximately $4.6 million and required 175 billion parameters. A standard computer vision model might achieve 92% accuracy on ImageNet with 25 million parameters. The data preprocessing phase typically takes 60% of a machine learning project time

In [58]:
def create_agent_prompt(question):
    """Create prompt that teaches the LLM to use tools"""
    
    tools_description = """Available Tools:
- calculator(expression): Calculate math expressions like "2+2" or "sqrt(16)"
- text_analyzer(text): Analyze text statistics 
- date_calculator(operation, days): Calculate dates like date_calculator("add", 30)
- number_extractor(text): Extract numbers from text
- counter(items): Count comma-separated items
- document_search(query): Search documents for information

To use a tool, write: TOOL_NAME(arguments)
Example: calculator("2 + 2") or document_search("machine learning")
"""

    prompt = f"""{tools_description}

Question: {question}

Think step by step and use tools if needed to answer this question. If you need to use a tool, write the tool call clearly like: TOOL_NAME(arguments)

Your response:"""

    return prompt

def agentic_rag(question, max_iterations=3):
    """Agentic RAG that can use tools iteratively"""
    
    print(f"🤖 AGENTIC RAG")
    print(f"Question: {question}")
    print("=" * 60)
    
    conversation_history = []
    
    for iteration in range(max_iterations):
        print(f"\n🔄 Iteration {iteration + 1}")
        print("-" * 30)
        
        # Create prompt with history
        if iteration == 0:
            prompt = create_agent_prompt(question)
        else:
            # Add previous results to context
            history_text = "\n".join(conversation_history)
            prompt = f"""Previous conversation:
{history_text}

Continue working on the question: {question}
Use tools if you need more information or calculations.

Your response:"""
        
        # Get LLM response
        print("🧠 LLM thinking...")
        llm_response = call_llm(prompt)
        print(f"LLM Response: {llm_response}")
        
        # Parse and execute tool calls
        tool_calls = parse_tool_call(llm_response)
        
        if tool_calls:
            print(f"\n🛠️ Using {len(tool_calls)} tools:")
            tool_results = []
            
            for tool_name, args in tool_calls:
                print(f"Executing: {tool_name}({args})")
                result = execute_tool(tool_name, args)
                print(f"Result: {result}")
                tool_results.append(f"{tool_name}({args}) -> {result}")
            
            # Add to conversation history
            conversation_history.append(f"LLM: {llm_response}")
            conversation_history.append(f"Tool Results: " + "; ".join(tool_results))
            
        else:
            # No tools used, this is likely the final answer
            print(f"\n✅ Final Answer: {llm_response}")
            return llm_response
    
    print(f"\n⚠️ Reached max iterations ({max_iterations})")
    return llm_response

# Test the agentic RAG
agentic_rag("What is the cost of training machine learning models and how much would it cost to train for 200 hours?")

🤖 AGENTIC RAG
Question: What is the cost of training machine learning models and how much would it cost to train for 200 hours?

🔄 Iteration 1
------------------------------
🧠 LLM thinking...
LLM Response: TOOL_NAME(arguments)
calculation_tool("cost_of_training_machine_learning_model", "machine learning")
calculation_tool("training_cost_200_hours", 200)

✅ Final Answer: TOOL_NAME(arguments)
calculation_tool("cost_of_training_machine_learning_model", "machine learning")
calculation_tool("training_cost_200_hours", 200)


'TOOL_NAME(arguments)\ncalculation_tool("cost_of_training_machine_learning_model", "machine learning")\ncalculation_tool("training_cost_200_hours", 200)'

In [59]:
# Test various scenarios that need different tools
scenarios = [
    {
        "question": "How many parameters does a typical neural network have, and what's the square root of that number?",
        "expected_tools": ["document_search", "calculator"]
    },
    {
        "question": "If I start training a model today and it takes 100 hours, what date will it finish?",
        "expected_tools": ["date_calculator", "document_search"]
    },
    {
        "question": "Extract all the numbers from information about machine learning costs",
        "expected_tools": ["document_search", "number_extractor"]
    },
    {
        "question": "Analyze the text about machine learning and count how many times different frameworks are mentioned",
        "expected_tools": ["document_search", "text_analyzer"]
    }
]

print("🧪 Testing Different Agent Scenarios")
print("=" * 60)

for i, scenario in enumerate(scenarios):
    print(f"\n🔬 Test {i+1}: {scenario['question']}")
    print(f"Expected tools: {scenario['expected_tools']}")
    print("=" * 50)
    
    result = agentic_rag(scenario['question'], max_iterations=2)
    print("\n" + "🔄" * 20)

🧪 Testing Different Agent Scenarios

🔬 Test 1: How many parameters does a typical neural network have, and what's the square root of that number?
Expected tools: ['document_search', 'calculator']
🤖 AGENTIC RAG
Question: How many parameters does a typical neural network have, and what's the square root of that number?

🔄 Iteration 1
------------------------------
🧠 LLM thinking...
LLM Response: TOOL_NAME(arguments)
TOOL_NAME(arguments) 2
TOOL_NAME(arguments) sqrt(2)

✅ Final Answer: TOOL_NAME(arguments)
TOOL_NAME(arguments) 2
TOOL_NAME(arguments) sqrt(2)

🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄🔄

🔬 Test 2: If I start training a model today and it takes 100 hours, what date will it finish?
Expected tools: ['date_calculator', 'document_search']
🤖 AGENTIC RAG
Question: If I start training a model today and it takes 100 hours, what date will it finish?

🔄 Iteration 1
------------------------------
🧠 LLM thinking...
LLM Response: TOOL_NAME(arguments) = calculator("add", 100)

To calculate how many days the mode

Number of requested results 3 is greater than number of elements in index 1, updating n_results = 1


LLM Response: I can help with that. To extract numbers from the given document, I'll use a combination of natural language processing (NLP) and regular expressions techniques.

First, let's analyze the document:

"Machine Learning Costs: The average cost per day for training a machine learning model ranges from $5,000 to $50,000 depending on the complexity of the task. However, for large-scale projects with multiple teams working simultaneously, costs can skyrocket up to $500,000 or more. Additionally, the time-consuming process of data preprocessing and feature engineering can extend hours beyond what was initially planned."

Now, let's use some tools to extract numbers from this text:

1. **Google's Ngram Viewer**: This tool can help us identify patterns in word frequencies over time. By analyzing the frequency of words like "machine learning", "costs", "average", and "range", we can infer that these are the main topics being discussed.
2. **TextBlob**: This API provides a set of mac

In [60]:
def analyze_agent_decisions():
    """Analyze when the agent chooses to use different tools"""
    
    test_questions = [
        "What is 25 multiplied by 48?",  # Should use calculator
        "How many words are in the machine learning text?",  # Should use text_analyzer
        "What date is 45 days from today?",  # Should use date_calculator  
        "What are the costs mentioned in the documents?",  # Should use document_search + number_extractor
        "Tell me about neural networks"  # Should use document_search only
    ]
    
    print("🔍 Agent Decision Making Analysis")
    print("=" * 50)
    
    for question in test_questions:
        print(f"\n❓ Question: {question}")
        print("-" * 40)
        
        # Create prompt and get initial response
        prompt = create_agent_prompt(question)
        response = call_llm(prompt)
        
        # Analyze what tools the agent wants to use
        tool_calls = parse_tool_call(response)
        
        print(f"Agent's reasoning: {response[:150]}...")
        print(f"Tools chosen: {[call[0] for call in tool_calls]}")
        
        if tool_calls:
            print("Tool execution:")
            for tool_name, args in tool_calls:
                result = execute_tool(tool_name, args)
                print(f"  {tool_name}({args}) -> {result[:100]}...")
        else:
            print("No tools used - direct answer")
        
        print("=" * 50)

analyze_agent_decisions()

🔍 Agent Decision Making Analysis

❓ Question: What is 25 multiplied by 48?
----------------------------------------


Number of requested results 3 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 3 is greater than number of elements in index 1, updating n_results = 1


Agent's reasoning: document_search("math multiplication")...
Tools chosen: ['document_search']
Tool execution:
  document_search("math multiplication") -> Found relevant documents:
Document 1 (relevance: 0.40): Machine learning algorithms require computat...

❓ Question: How many words are in the machine learning text?
----------------------------------------
Agent's reasoning: document_search("machine learning")...
Tools chosen: ['document_search']
Tool execution:
  document_search("machine learning") -> Found relevant documents:
Document 1 (relevance: 0.74): Machine learning algorithms require computat...

❓ Question: What date is 45 days from today?
----------------------------------------


Number of requested results 3 is greater than number of elements in index 1, updating n_results = 1


Agent's reasoning: date_calculator("subtract", 45)
 

or
TOOL_NAME(arguments): date_calculator("add", days=45)...
Tools chosen: ['date_calculator', 'date_calculator']
Tool execution:
  date_calculator("subtract", 45) -> Today - 45 days = 2025-04-07...
  date_calculator("add", days=45) -> Today + 0 days = 2025-05-22...

❓ Question: What are the costs mentioned in the documents?
----------------------------------------
Agent's reasoning: document_search("costs")...
Tools chosen: ['document_search']
Tool execution:
  document_search("costs") -> Found relevant documents:
Document 1 (relevance: 0.55): Machine learning algorithms require computat...

❓ Question: Tell me about neural networks
----------------------------------------


Number of requested results 3 is greater than number of elements in index 1, updating n_results = 1


Agent's reasoning: TOOL_NAME(arguments)
calculator("2 + 2") 
document_search("machine learning")
text_analyzer("neural networks")
counter(["artificial intelligence", "ma...
Tools chosen: ['calculator', 'document_search', 'text_analyzer', 'counter', 'date_calculator']
Tool execution:
  calculator("2 + 2") -> Result: 4...
  document_search("machine learning") -> Found relevant documents:
Document 1 (relevance: 0.74): Machine learning algorithms require computat...
  text_analyzer("neural networks") -> Text Analysis:
- Words: 2
- Characters: 15
- Sentences: 1
- Average word length: 7.5
- Longest word:...
  counter(["artificial intelligence", "machine learning"]) -> Item counts:
- "machine learning"]: 1
- ["artificial intelligence": 1
...
  date_calculator("add", 30) -> Today + 30 days = 2025-06-21...


# Multilingual embeddings


https://huggingface.co/intfloat/multilingual-e5-large